In [ ]:
# Created by Stephen Price on March 18th, 2026
# Released under the Apache 2.0 License

# Evaluating Quality of Composition Mappings

## Mapped/Unmapped String Identification

This code evaluates all of the existing composition mappings (wt.% and at%), executes them on existing composition strings in the dataset, identify all mapped and unmapped strings for majority, secondary, and tertiary compositions

In [ ]:
import json
import re
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

from pymatgen.core.composition import Composition
from pymatgen.core.periodic_table import Element


def map_powder_compositions_and_report_jupyter(
    exp_json_path: str = "../HUGO-CS/Dataset/AlternateVersions/Extracted-6-10-25_gt_remapped_swapped_deleted_filled_sorted.json",
    replace_json_path: str = "../HUGO-CS/prompts/regExReplace2.json",
    section: str = "preSprayedProperties",
    decimals: int = 3,
    map_column_width: int = 125,   
    max_json_chars: Optional[int] = None,  
) -> Dict[str, Any]:
    

    WS_RE = re.compile(r"\s+")

    def _norm_key(s: str) -> str:
        return WS_RE.sub(" ", s.strip()).lower()

    def _coerce_to_dict(val: Any) -> Optional[Dict[str, Any]]:
        if isinstance(val, dict):
            return val
        if isinstance(val, str):
            try:
                parsed = json.loads(val)
                return parsed if isinstance(parsed, dict) else None
            except json.JSONDecodeError:
                return None
        return None

    def _canonical_json(d: Dict[str, Any]) -> str:
        return json.dumps(d, ensure_ascii=False, sort_keys=True)

    def atomic_map_to_wtjson_general(full_atomic_map: Dict[str, Any], decimals_local: int) -> str:
        suffix = "_Percentage"

        
        at_pct: Dict[str, float] = {}
        for k, v in full_atomic_map.items():
            if not isinstance(k, str) or not k.endswith(suffix):
                continue
            base = k[:-len(suffix)]
            if "_" not in base:
                continue
            _, el = base.rsplit("_", 1)
            try:
                at_pct[el] = float(v)
            except (TypeError, ValueError):
                continue

        comp = Composition({el: pct / 100.0 for el, pct in at_pct.items()})
        mol_dict = comp.get_el_amt_dict()
        mass_contrib = {el: amt * Element(el).atomic_mass for el, amt in mol_dict.items()}
        total_mass = float(sum(mass_contrib.values())) or 1.0

        pct_keys: List[str] = []
        pct_vals: List[float] = []
        for k in full_atomic_map.keys():
            if isinstance(k, str) and k.endswith(suffix):
                base = k[:-len(suffix)]
                if "_" in base:
                    _, el = base.rsplit("_", 1)
                    if el in mass_contrib:
                        pct_keys.append(k)
                        pct_vals.append(float(mass_contrib[el]) / total_mass * 100.0)

        scale = 10 ** decimals_local
        rounded = [round(x, decimals_local) for x in pct_vals]
        units = [int(round(x * scale)) for x in rounded]
        target = 100 * scale
        diff = target - sum(units)

        errors = [(pct_vals[i] - rounded[i], i) for i in range(len(pct_vals))]

        if units and diff != 0:
            if diff > 0:
                errors.sort(reverse=True)
                for t in range(diff):
                    units[errors[t % len(errors)][1]] += 1
            else:
                errors.sort()
                for t in range(-diff):
                    idx = errors[t % len(errors)][1]
                    if units[idx] > 0:
                        units[idx] -= 1
                    else:
                        for _, j in errors:
                            if units[j] > 0:
                                units[j] -= 1
                                break

        final_vals = [u / scale for u in units]

        wt_map = dict(full_atomic_map)
        fmt = f"{{:.{decimals_local}f}}"
        for k, v in zip(pct_keys, final_vals):
            wt_map[k] = fmt.format(v)

        return _canonical_json(wt_map)

    def flatten_experiments_from_articles(articles: Any) -> List[Dict[str, Any]]:
        all_exps: List[Dict[str, Any]] = []
        if not isinstance(articles, list):
            return all_exps

        for art in articles:
            if not isinstance(art, dict):
                continue
            ext = art.get("extractedText", {})
            if isinstance(ext, str):
                try:
                    ext = json.loads(ext)
                except json.JSONDecodeError:
                    ext = {}

            exps: List[Any] = []
            if isinstance(ext, dict):
                exps = ext.get("Experiments", []) or []
            elif isinstance(ext, list):
                for item in ext:
                    if isinstance(item, dict):
                        exps.extend(item.get("Experiments", []) or [])

            for e in exps:
                if isinstance(e, dict):
                    all_exps.append(e)

        return all_exps

    with open(exp_json_path, "r", encoding="utf-8") as f:
        articles = json.load(f)

    with open(replace_json_path, "r", encoding="utf-8") as f:
        replace_dict = json.load(f)

    wt_map = replace_dict.get("chemicalCompositionMapping", {}) or {}
    atomic_map = replace_dict.get("chemicalCompositionMappingAtomic", {}) or {}

    wt_lower = {_norm_key(k): v for k, v in wt_map.items()} if isinstance(wt_map, dict) else {}
    at_lower = {_norm_key(k): v for k, v in atomic_map.items()} if isinstance(atomic_map, dict) else {}

    all_experiments = flatten_experiments_from_articles(articles)

    features = [
        "Majority_Powder_Chemical_Composition",
        "Secondary_Powder_Chemical_Composition",
        "Tertiary_Powder_Chemical_Composition",
    ]

    mapped_counts = {f: Counter() for f in features}
    unmapped_counts = {f: Counter() for f in features}
    mapped_to: Dict[str, Dict[str, str]] = {f: {} for f in features}

    for exp in all_experiments:
        props = exp.get(section, {})
        if not isinstance(props, dict):
            continue

        for feature in features:
            raw = props.get(feature)
            if not isinstance(raw, str):
                continue

            raw_str = raw
            raw_norm = _norm_key(raw_str)

            d0 = _coerce_to_dict(raw_str)
            if d0 is not None:
                mapped_json = _canonical_json(d0)
                mapped_counts[feature][raw_str] += 1
                mapped_to[feature][raw_str] = mapped_json
                continue

            if raw_norm in wt_lower:
                rep = wt_lower[raw_norm]
                rep_dict = _coerce_to_dict(rep)
                if rep_dict is not None:
                    mapped_json = _canonical_json(rep_dict)
                    props[feature] = mapped_json
                    mapped_counts[feature][raw_str] += 1
                    mapped_to[feature][raw_str] = mapped_json
                    continue

            if raw_norm in at_lower:
                full_map_val = at_lower[raw_norm]
                full_map_dict = _coerce_to_dict(full_map_val)
                if full_map_dict is not None:
                    try:
                        mapped_json = atomic_map_to_wtjson_general(full_map_dict, decimals)
                        props[feature] = mapped_json
                        mapped_counts[feature][raw_str] += 1
                        mapped_to[feature][raw_str] = mapped_json
                        continue
                    except Exception as e:
                        print(f"Warning: atomic→weight conversion failed for {feature}={raw_str!r}: {e}")

            unmapped_counts[feature][raw_str] += 1

    def _maybe_trunc(s: str) -> str:
        if max_json_chars is None or len(s) <= max_json_chars:
            return s
        return s[: max_json_chars - 3] + "..."

    def _print_mapped(title: str, counter: Counter, mapping: Dict[str, str]) -> None:
        print(title)
        for original, cnt in counter.most_common():
            mapped_json = mapping.get(original, "")
            left = f"  {cnt:>6}  {original!r}"
            pad = " " * max(1, map_column_width - len(left))
            print(f"{left}{pad} -> {_maybe_trunc(mapped_json)}")
        print(f"  Total unique: {len(counter)}\n")

    def _print_unmapped(title: str, counter: Counter) -> None:
        print(title)
        for original, cnt in counter.most_common():
            print(f"  {cnt:>6}  {original!r}")
        print(f"  Total unique: {len(counter)}\n")

    _print_mapped(
        "1) Majority_Powder_Chemical_Composition MAPPED (original string  ->  mapped JSON)",
        mapped_counts[features[0]],
        mapped_to[features[0]],
    )
    _print_unmapped(
        "2) Majority_Powder_Chemical_Composition UNMAPPED (remaining)",
        unmapped_counts[features[0]],
    )

    _print_mapped(
        "3) Secondary_Powder_Chemical_Composition MAPPED (original string  ->  mapped JSON)",
        mapped_counts[features[1]],
        mapped_to[features[1]],
    )
    _print_unmapped(
        "4) Secondary_Powder_Chemical_Composition UNMAPPED (remaining)",
        unmapped_counts[features[1]],
    )

    _print_mapped(
        "5) Tertiary_Powder_Chemical_Composition MAPPED (original string  ->  mapped JSON)",
        mapped_counts[features[2]],
        mapped_to[features[2]],
    )
    _print_unmapped(
        "6) Tertiary_Powder_Chemical_Composition UNMAPPED (remaining)",
        unmapped_counts[features[2]],
    )

    return {
        "all_experiments": all_experiments, 
        "mapped_counts": mapped_counts,
        "unmapped_counts": unmapped_counts,
        "mapped_to": mapped_to,
        "features": features,
        "section": section,
        "decimals": decimals,
        "exp_json_path": exp_json_path,
        "replace_json_path": replace_json_path,
    }



result = map_powder_compositions_and_report_jupyter(
    exp_json_path="../HUGO-CS/Dataset/AlternateVersions/Extracted-6-10-25_gt_remapped_swapped_deleted_filled_sorted.json",
    replace_json_path="../HUGO-CS/prompts/regExReplace2.json",
    section="preSprayedProperties",
    decimals=3,
)


1) Majority_Powder_Chemical_Composition MAPPED (original string  ->  mapped JSON)
      88  'Cu'                                                                                                                -> {"Majority_Powder_Ag_Percentage": "", "Majority_Powder_Al_Percentage": "", "Majority_Powder_As_Percentage": "", "Majority_Powder_B_Percentage": "", "Majority_Powder_Ba_Percentage": "", "Majority_Powder_Be_Percentage": "", "Majority_Powder_C_Percentage": "", "Majority_Powder_Ca_Percentage": "", "Majority_Powder_Cd_Percentage": "", "Majority_Powder_Cl_Percentage": "", "Majority_Powder_Co_Percentage": "", "Majority_Powder_Cr_Percentage": "", "Majority_Powder_Cu_Percentage": "100.0", "Majority_Powder_Fe_Percentage": "", "Majority_Powder_Ga_Percentage": "", "Majority_Powder_Gd_Percentage": "", "Majority_Powder_Ge_Percentage": "", "Majority_Powder_H_Percentage": "", "Majority_Powder_Hf_Percentage": "", "Majority_Powder_In_Percentage": "", "Majority_Powder_K_Percentage": "", "Majority_

## Fully Composition Representation
This code evaluates the weight percent, atomic percent, and imposition mapping to identify strings that don't equal 100%, proposing a normalized verion to 100%, and giving a new composition dictionary corrected

In [3]:
import json
import math
import re
from pathlib import Path
from typing import Any, Dict, Tuple, List

REPLACE_JSON = "../HUGO-CS/prompts/regExReplace2.json"

MAPPING_KEYS = [
    "chemicalCompositionMappingAtomic",
    "chemicalCompositionMapping",
    "chemicalCompositionImputation",
]

ABS_TOL = 1e-3
NUM_RE = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def parse_number(x):
    if x is None:
        return 0.0, True
    if isinstance(x, (int, float)):
        return float(x), True
    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return 0.0, True
        m = NUM_RE.search(s)
        if not m:
            return 0.0, False
        try:
            return float(m.group(0)), True
        except ValueError:
            return 0.0, False
    return 0.0, False

def normalize_2dp_sum100(values):
    if not values:
        return []
    total = sum(values)
    if total <= 0:
        return [0.0 for _ in values]

    raw = [v * 100.0 / total for v in values]
    rounded = [round(x, 3) for x in raw]

    mills = [int(round(x * 1000)) for x in rounded]
    current = sum(mills)
    target = 100000
    diff = target - current

    errors = []
    for i in range(len(values)):
        errors.append(raw[i] - rounded[i])

    if diff > 0:
        order = sorted(range(len(values)), key=lambda i: errors[i], reverse=True)
        for k in range(diff):
            mills[order[k % len(values)]] += 1
    elif diff < 0:
        order = sorted(range(len(values)), key=lambda i: errors[i])
        for k in range(-diff):
            idx = order[k % len(values)]
            if mills[idx] > 0:
                mills[idx] -= 1
            else:
                for j in order:
                    if mills[j] > 0:
                        mills[j] -= 1
                        break

    return [m / 1000.0 for m in mills]

def strip_pct_field_name(field_name):
    return field_name.replace("Majority_Powder_", "").replace("_Percentage", "")

def build_summary(fields, values):
    parts = []
    for f, val in zip(fields, values):
        if val == 0:
            continue
        parts.append(f"{strip_pct_field_name(f)}: {val:.3f}")
    return ", ".join(parts) if parts else "(all zero)"

def strip_trailing_zeros_keep_one_decimal(s):
    if "e" in s or "E" in s:
        return s
    if "." not in s:
        return s + ".0"
    left, right = s.split(".", 1)
    right = right.rstrip("0")
    if right == "":
        right = "0"
    return left + "." + right

def number_to_quoted_string(x):
    if isinstance(x, bool):
        return str(x)
    if isinstance(x, int):
        return strip_trailing_zeros_keep_one_decimal(f"{x}.0")
    if isinstance(x, float):
        return strip_trailing_zeros_keep_one_decimal(str(x))
    return str(x)

def check_one_mapping(mapping_key, mapping):
    print(f"\n=== Checking: {mapping_key} ===")
    issues_found = False

    for material_key, v in mapping.items():
        if not isinstance(v, dict):
            print(f"[NOT_A_DICT] {material_key!r} -> {type(v).__name__}")
            issues_found = True
            continue

        pct_fields = [k for k in v.keys() if k.endswith("_Percentage")]
        if not pct_fields:
            print(f"[NO_PERCENT_FIELDS] {material_key!r}")
            issues_found = True
            continue

        total = 0.0
        bad_fields = []
        nums_by_field = {}

        for field_name in pct_fields:
            num, ok = parse_number(v.get(field_name))
            nums_by_field[field_name] = num
            if not ok:
                bad_fields.append(f"{field_name}={v.get(field_name)!r}")
            total += num

        if bad_fields or not math.isclose(total, 100.0, abs_tol=ABS_TOL):
            issues_found = True

            reason = []
            if bad_fields:
                reason.append("unparsable: " + ", ".join(bad_fields))
            reason.append(f"sum={total:.6g}")
            print(f"[BAD_SUM] {material_key!r} -> " + " | ".join(reason))

            ordered_fields = pct_fields[:]
            pre_values = [nums_by_field[f] for f in ordered_fields]
            normalized = normalize_2dp_sum100(pre_values)

            pre_summary = build_summary(ordered_fields, pre_values)
            post_summary = build_summary(ordered_fields, normalized)

            print("  Pre:  " + pre_summary)
            print("  Post: " + post_summary)

            updated = dict(v)
            for f, val in zip(ordered_fields, normalized):
                val_rounded = round(val, 3)
                updated[f] = strip_trailing_zeros_keep_one_decimal(f"{val_rounded:.3f}")

            for k in list(updated.keys()):
                if isinstance(updated[k], (int, float)) and not isinstance(updated[k], bool):
                    updated[k] = number_to_quoted_string(updated[k])

            print(json.dumps(updated, indent=2))

    if not issues_found:
        print("No issues found")

def main():
    path = Path(REPLACE_JSON)
    with path.open("r", encoding="utf-8") as f:
        replace_dict = json.load(f)

    for key in MAPPING_KEYS:
        mapping = replace_dict.get(key)
        if not isinstance(mapping, dict):
            print(f"\n=== Checking: {key} ===")
            print(f"[NOT_A_DICT_MAPPING] {key!r} -> {type(mapping).__name__}")
            continue
        check_one_mapping(key, mapping)

main()



=== Checking: chemicalCompositionMappingAtomic ===
No issues found

=== Checking: chemicalCompositionMapping ===
[NOT_A_DICT] '' -> str

=== Checking: chemicalCompositionImputation ===
No issues found


## LLM Verification of Composition Mapping:

This code uses an LLM to evaluate all of the composition mappings (weight, atomic, and imputation) to identify any string mappings with unusual, or expected coverage mappings

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

from openai import OpenAI



REPLACE_JSON = "../HUGO-CS/prompts/regExReplace2.json"

WEIGHT_KEY = "chemicalCompositionMapping"
ATOMIC_KEY = "chemicalCompositionMappingAtomic"
IMPUTE_KEY = "chemicalCompositionImputation"

FILE_JOBS = [
    (WEIGHT_KEY, "reviews/weightComposition_review.json"),
    (ATOMIC_KEY, "reviews/atomicComposition_review.json"),
    (IMPUTE_KEY, "reviews/compositionImputation_review.json"),
]

OPENAI_TOKEN_PATH = (
    "../openaiToken.txt"
)

MODEL = "o4-mini"
REASONING_EFFORT = "high"
MAX_WORKERS = 8
MAX_RETRIES = 5
BACKOFF_SECONDS = 2.0


def load_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Any, path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
        f.write("\n")
    os.replace(tmp, path)


def clean_json_fence(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        t = t.replace("```json", "").replace("```", "").strip()
    return t


def try_parse_json(text: str) -> Optional[Dict[str, Any]]:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


def run_llm_assessment(client: OpenAI, key: str, value: Any) -> Dict[str, Any]:
    system = (
        "You are a meticulous materials-science data QA reviewer. "
        "You will be given a mapping entry: a KEY and its VALUE (JSON). "
        "Do not worry if the composition sums to 100; some entries were normalized to 100% "
        "and minor rounding/normalization irregularities are acceptable. "
        "Your task is to judge whether the mapping is correct and internally consistent GIVEN THE KEY TEXT.\n\n"
        "Important interpretation rules:\n"
        "- In the VALUE dictionary, an empty string \"\" for an element percentage field means "
        "the element is not included / not reported for that entry. This is normal and not a problem.\n"
        "- Do NOT apply external typical-spec knowledge (e.g., typical impurity ranges) to mark entries wrong. "
        "Evaluate correctness primarily against what is explicitly stated in the KEY.\n"
        "- Inequalities in the KEY are allowed to be represented as exact numeric values in the mapping. "
        "Treat '<', '<=', 'max', 'maximum', 'up to' as upper bounds: a mapped value equal to the bound is acceptable "
        "(e.g., KEY '<0.05' may map to 0.05; KEY 'O max 0.2' may map to 0.2). "
        "Similarly, treat '>', '>=', 'min', 'minimum', 'at least' as lower bounds: a mapped value equal to the bound is acceptable.\n"
        "- Purity language and remainder handling: if the KEY states a base element purity (e.g., 'Ti 99.9 wt%') "
        "and also lists impurity limits (e.g., 'O max 0.2 wt%'), do NOT mark the mapping wrong if it assigns the impurity at its limit "
        "and reduces the base element accordingly to keep totals consistent (e.g., Ti 99.8 and O 0.2 is acceptable). "
        "Only flag a problem if the mapping clearly contradicts the KEY (e.g., assigns an impurity above its stated maximum, "
        "or assigns a required minimum below its minimum).\n"
        "- Do NOT require the mapping to preserve the exact written purity figure for the base element when the KEY also provides "
        "impurity maxima/minima; those constraints can be simultaneously satisfied by allocating impurities and letting the base element be the remainder.\n\n"
        "Return ONLY valid JSON with this schema:\n"
        "{\n"
        '  \"rating\": 1|2|3,\n'
        '  \"verdict\": \"Fully Correct\" | \"Potentially incorrect\" | \"Definitely wrong\",\n'
        '  \"explanation\": \"Concise but specific justification. Mention what to check or what is wrong.\"\n'
        "}\n\n"
        "Rating definitions:\n"
        "1 = Fully Correct: no issues found.\n"
        "2 = Potentially incorrect: plausible but uncertain; explain why and what would confirm.\n"
        "3 = Definitely wrong: clear contradictions or invalid mapping; explain why.\n\n"
        "Be strict and concrete. If something is ambiguous, choose 2."
    )


    user = (
        "Evaluate this mapping entry.\n\n"
        f"KEY:\n{key}\n\n"
        "VALUE (JSON):\n"
        f"{json.dumps(value, indent=2, ensure_ascii=False)}\n"
    )

    last_err: Optional[Exception] = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                reasoning_effort=REASONING_EFFORT
            )
            raw = resp.choices[0].message.content or ""
            tokens = int(getattr(resp.usage, "total_tokens", 0) or 0)

            cleaned = clean_json_fence(raw)
            parsed = try_parse_json(cleaned)

            if parsed is None:
                return {
                    "rating": None,
                    "verdict": None,
                    "explanation": None,
                    "raw": cleaned,
                    "tokens": tokens,
                    "error": "invalid_json_from_model",
                }

            rating = parsed.get("rating")
            verdict = parsed.get("verdict")
            explanation = parsed.get("explanation")

            if rating not in (1, 2, 3):
                rating = None

            return {
                "rating": rating,
                "verdict": verdict,
                "explanation": explanation,
                "raw": cleaned,
                "tokens": tokens,
            }

        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(BACKOFF_SECONDS * attempt)
                continue
            raise RuntimeError(f"LLM call failed after {MAX_RETRIES} attempts: {e}") from e

    raise RuntimeError(f"Unexpected failure: {last_err}")


def assess_file(client: OpenAI, mapping: Dict[str, Any], group_key: str, output_json: str) -> None:
    if not isinstance(mapping, dict):
        raise TypeError(f"Expected group {group_key} to be a JSON object (dict), got {type(mapping).__name__}")

    results: List[Dict[str, Any]] = []
    token_total = 0
    keys = [k for k in mapping.keys() if k != ""]

    print(f"Loaded {len(mapping)} entries from {group_key}. Assessing {len(keys)} (excluding empty-string key).")

    def worker(idx: int, k: str) -> Dict[str, Any]:
        v = mapping[k]
        assessment = run_llm_assessment(client, k, v)
        row = {
            "idx": idx,
            "key": k,
            "value": v,
            "rating": assessment.get("rating"),
            "verdict": assessment.get("verdict"),
            "explanation": assessment.get("explanation"),
            "tokens": assessment.get("tokens"),
            "raw": assessment.get("raw"),
        }
        if "error" in assessment:
            row["error"] = assessment["error"]
        return row

    jobs: List[Tuple[int, str]] = list(enumerate(keys, start=1))
    job_iter = iter(jobs)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {}

        # initial fill
        while len(futures) < MAX_WORKERS:
            try:
                idx, k = next(job_iter)
            except StopIteration:
                break
            fut = pool.submit(worker, idx, k)
            futures[fut] = (idx, k)

        while futures:
            done, _ = wait(futures, return_when=FIRST_COMPLETED)
            for fut in done:
                idx, k = futures.pop(fut)
                row = fut.result()

                token_total += int(row.get("tokens", 0) or 0)
                results.append(row)

                rating = row["rating"]
                verdict = row["verdict"]
                expl = row["explanation"] or row.get("raw", "")
                expl_preview = expl.replace("\n", " ").strip()
                if len(expl_preview) > 220:
                    expl_preview = expl_preview[:220] + "..."

                print(f"[{row['idx']}/{len(keys)}] rating={rating} verdict={verdict} tokens_total={token_total}")
                print(f"  KEY: {row['key']}")
                print(f"  NOTE: {expl_preview}")
                print("-" * 80)

            while len(futures) < MAX_WORKERS:
                try:
                    idx, k = next(job_iter)
                except StopIteration:
                    break
                fut = pool.submit(worker, idx, k)
                futures[fut] = (idx, k)

    results_sorted = sorted(results, key=lambda r: r.get("idx", 10**18))

    out_obj = {
        "input_file": group_key,
        "model": MODEL,
        "reasoning_effort": REASONING_EFFORT,
        "total_tokens": token_total,
        "results": results_sorted,
    }
    save_json(out_obj, output_json)
    print(f"Saved {len(results_sorted)} assessments to {output_json}. Total tokens: {token_total}")


def main() -> None:
    api_key = Path(OPENAI_TOKEN_PATH).read_text(encoding="utf-8").strip()
    os.environ["OPENAI_API_KEY"] = api_key
    client = OpenAI(api_key=api_key)

    big = load_json(REPLACE_JSON)
    if not isinstance(big, dict):
        raise TypeError(f"Expected {REPLACE_JSON} to be a JSON object (dict), got {type(big).__name__}")

    for group_key, output_json in FILE_JOBS:
        mapping = big.get(group_key)
        if mapping is None:
            raise KeyError(f"Missing required group {group_key!r} in {REPLACE_JSON}")
        assess_file(client, mapping, group_key, output_json)


if __name__ == "__main__":
    main()


Loaded 867 entries from chemicalCompositionMapping. Assessing 866 (excluding empty-string key).
[8/866] rating=1 verdict=Fully Correct tokens_total=1385
  KEY: 99.9 % pure copper
  NOTE: The mapping assigns Cu=99.9% and lumps the 0.1% remainder as ‘Other,’ matching the 99.9% pure copper specification with allowable impurities.
--------------------------------------------------------------------------------
[6/866] rating=1 verdict=Fully Correct tokens_total=2787
  KEY: 99.9 wt% Cu
  NOTE: The KEY specifies 99.9 wt% Cu. The mapping assigns Cu = 99.9 and allocates the remaining 0.1 wt% to “Other,” which is consistent with the specification and sums to 100 wt%.
--------------------------------------------------------------------------------
[1/866] rating=1 verdict=Fully Correct tokens_total=4174
  KEY: Ni 80 wt%, Cr 20 wt%
  NOTE: The mapping assigns Ni = 80 wt% and Cr = 20 wt% exactly as specified in the KEY, with no other elements reported. This matches the KEY perfectly.
-------------

## Comparing Labeled Primary Element to Computed Primary Element

This code evaluates the largest element present in the mapped composition to the labeled primary element of the powder to identify any inconcistencies.

In [ ]:
import json
import re
from collections import Counter
from typing import Any, Dict, List, Optional, Tuple

# Allowed primary-element labels
ELEMENTS_ALLOWED = [
    "Ag","Al","As","B","Ba","Be","C","Ca","Cd","Cl","Co","Cr","Cu","Fe","Ga","Gd","Ge","H","Hf","In","K","La",
    "Mg","Mn","Mo","N","Na","Nb","Ni","O","P","Pb","Pd","S","Sb","Sc","Si","Sn","Sr","Ta","Ti","U","V","W","Y",
    "Zn","Zr","Other"
]

WS_RE = re.compile(r"\s+")

def evaluate_powder_primary_element_checks_jupyter(
    exp_json_path: str = "../HUGO-CS/Dataset/AlternateVersions/Extracted-6-10-25_gt_remapped_swapped_deleted_filled_sorted_powderClass_grouped_imputed_blended_treatment.json",
    section: str = "preSprayedProperties",
    allowed_elements: Optional[List[str]] = None,
    align_width: int = 30,
) -> Dict[str, Any]:
    """
    Evaluates:
      Check 1) If Multiple_Powders_Binary == False, Secondary/Tertiary primary elements must be blank "".
      Check 2) If any primary element is non-blank, it must be in allowed_elements.
      Check 3) If primary element is non-blank AND composition field is a mapped JSON string, the largest-% element in
               that JSON must equal the labeled primary element. (If labeled "Other", this check passes by default.)

    Prints 18 lists total:
      - For each check (1..3): 6 lists (PASS/FAIL for Majority, Secondary, Tertiary).
        (PASS=Mapped, FAIL=Unmapped)

    Returns a dict with counters and bookkeeping.
    """

    allowed = set(allowed_elements or ELEMENTS_ALLOWED)

    def _norm_space(s: str) -> str:
        return WS_RE.sub(" ", s.strip())

    def _blankish(x: Any) -> bool:
        return x is None or (isinstance(x, str) and (x.strip() == "" or x.strip() == "[V] Not Reported"))

    def _as_str(x: Any) -> Optional[str]:
        return x if isinstance(x, str) else None

    def _coerce_json_dict(s: str) -> Optional[Dict[str, Any]]:
        try:
            obj = json.loads(s)
            return obj if isinstance(obj, dict) else None
        except Exception:
            return None

    def _extract_filename(art: Dict[str, Any]) -> str:
        for k in (
            "fileName", "filename", "file_name",
            "pdfFileName", "pdf_filename", "pdf_name",
            "sourceFile", "source_file",
            "path", "filepath", "filePath",
            "pdf", "pdfPath", "pdf_path",
            "document", "doc",
        ):
            v = art.get(k)
            if isinstance(v, str) and v.strip():
                return v.strip()
        return "(unknown file)"

    def _flatten_experiments_from_articles(articles: Any) -> List[Dict[str, Any]]:
        all_exps: List[Dict[str, Any]] = []
        if not isinstance(articles, list):
            return all_exps

        for art in articles:
            if not isinstance(art, dict):
                continue

            src_file = _extract_filename(art)

            ext = art.get("extractedText", {})
            if isinstance(ext, str):
                try:
                    ext = json.loads(ext)
                except json.JSONDecodeError:
                    ext = {}

            exps: List[Any] = []
            if isinstance(ext, dict):
                exps = ext.get("Experiments", []) or []
            elif isinstance(ext, list):
                for item in ext:
                    if isinstance(item, dict):
                        exps.extend(item.get("Experiments", []) or [])

            for e in exps:
                if isinstance(e, dict):
                    if "__source_filename" not in e:
                        e["__source_filename"] = src_file
                    all_exps.append(e)

        return all_exps

    def _extract_max_element_from_comp(comp_dict: Dict[str, Any]) -> Optional[Tuple[str, float]]:
        best_el: Optional[str] = None
        best_val: float = float("-inf")

        for k, v in comp_dict.items():
            if not isinstance(k, str) or not k.endswith("_Percentage"):
                continue
            base = k[:-len("_Percentage")]
            if "_" not in base:
                continue
            _, el = base.rsplit("_", 1)

            try:
                pct = float(v)
            except Exception:
                continue

            if pct > best_val:
                best_val = pct
                best_el = el

        if best_el is None:
            return None
        return best_el, best_val

    def _elements_from_mpc(s: str) -> List[str]:
        # If starts with MPC:, parse comma-separated element symbols after it.
        # e.g. "MPC: Co, Cr, Fe, Mn, Ni" -> ["Co","Cr","Fe","Mn","Ni"]
        tail = s.split(":", 1)[1]
        return [t.strip() for t in tail.split(",") if t.strip()]

    def _element_pct_map_from_comp(comp_dict: Dict[str, Any]) -> Dict[str, float]:
        out: Dict[str, float] = {}
        for k, v in comp_dict.items():
            if not isinstance(k, str) or not k.endswith("_Percentage"):
                continue
            base = k[:-len("_Percentage")]
            if "_" not in base:
                continue
            _, el = base.rsplit("_", 1)
            try:
                out[el] = float(v)
            except Exception:
                continue
        return out

    def _print_pass_fail(title: str, passed: Counter, failed: Counter, checked: int, skipped: int = 0) -> None:
        print(title)
        meta = f"  Checked: {checked}"
        if skipped:
            meta += f" | Skipped: {skipped}"
        print(meta)

        def _print_counter_block(label: str, c: Counter) -> None:
            print(f"  {label}:")
            if not c:
                print("    (none)")
                return
            for val, cnt in c.most_common():
                print(f"    {cnt:>6}  {val}")

        _print_counter_block("PASS (Mapped)", passed)
        _print_counter_block("FAIL (Unmapped)", failed)
        print()

    with open(exp_json_path, "r", encoding="utf-8") as f:
        articles = json.load(f)
    all_experiments = _flatten_experiments_from_articles(articles)

    levels = [
        ("Majority",  "Majority_Powder_Primary_Element",  "Majority_Powder_Chemical_Composition"),
        ("Secondary", "Secondary_Powder_Primary_Element", "Secondary_Powder_Chemical_Composition"),
        ("Tertiary",  "Tertiary_Powder_Primary_Element",  "Tertiary_Powder_Chemical_Composition"),
    ]

    checks: Dict[int, Dict[str, Dict[str, Any]]] = {}
    for cid in (1, 2, 3):
        checks[cid] = {}
        for lvl, _, _ in levels:
            checks[cid][lvl] = {"passed": Counter(), "failed": Counter(), "checked": 0, "skipped": 0}

    check3_fail_files: Dict[str, Dict[str, set]] = {lvl: {} for (lvl, _, _) in levels}

    for exp in all_experiments:
        props = exp.get(section, {})
        if not isinstance(props, dict):
            continue

        mpb = props.get("Multiple_Powders_Binary")

        # ---------- Check 1 ----------
        # If Multiple_Powders_Binary == False, Secondary/Tertiary primary elements must be blank "".
        if (mpb is False) or (isinstance(mpb, str) and mpb.strip().lower() == "false"):
            # Majority: N/A bucket
            checks[1]["Majority"]["checked"] += 1
            checks[1]["Majority"]["passed"]["N/A"] += 1

            for lvl, primary_field, _ in levels[1:]:  # Secondary + Tertiary
                val = props.get(primary_field)
                checks[1][lvl]["checked"] += 1
                if _blankish(val):
                    checks[1][lvl]["passed"]['""'] += 1
                else:
                    s = _as_str(val)
                    display = repr(s) if s is not None else repr(val)
                    checks[1][lvl]["failed"][display] += 1
        else:
            for lvl, _, _ in levels:
                checks[1][lvl]["skipped"] += 1

        # ---------- Check 2 ----------
        # If non-blank, primary element must be in allowed list.
        # If starts with MPC:, verify all listed elements are real (in allowed, excluding "Other").
        for lvl, primary_field, _ in levels:
            val = props.get(primary_field)
            if _blankish(val):
                checks[2][lvl]["skipped"] += 1
                continue

            s = _as_str(val)
            checks[2][lvl]["checked"] += 1
            if s is None:
                checks[2][lvl]["failed"][repr(val)] += 1
                continue

            s_clean = _norm_space(s)

            if s_clean.lower().startswith("mpc:"):
                elems = _elements_from_mpc(s_clean)
                invalid = [e for e in elems if e not in allowed or e == "Other"]
                if not invalid and elems:
                    checks[2][lvl]["passed"][repr(s_clean)] += 1
                else:
                    checks[2][lvl]["failed"][f"{repr(s_clean)}"] += 1
            else:
                if s_clean in allowed:
                    checks[2][lvl]["passed"][repr(s_clean)] += 1
                else:
                    checks[2][lvl]["failed"][repr(s_clean)] += 1

        # ---------- Check 3 ----------
        # If non-blank primary AND composition is mapped JSON:
        #   - If primary starts with MPC:, check all those elements have composition > 1%
        #   - Else: max element must match primary (Other passes).
        for lvl, primary_field, comp_field in levels:
            primary_val = props.get(primary_field)
            comp_val = props.get(comp_field)

            if _blankish(primary_val):
                checks[3][lvl]["skipped"] += 1
                continue

            primary_str = _as_str(primary_val)
            if primary_str is None:
                checks[3][lvl]["checked"] += 1
                checks[3][lvl]["failed"][repr(primary_val)] += 1
                continue

            comp_str = _as_str(comp_val)
            if comp_str is None:
                checks[3][lvl]["skipped"] += 1
                continue

            comp_dict = _coerce_json_dict(comp_str)
            if comp_dict is None:
                checks[3][lvl]["skipped"] += 1
                continue

            primary_clean = _norm_space(primary_str)
            src_file = exp.get("__source_filename", "(unknown file)")

            if primary_clean.lower().startswith("mpc:"):
                checks[3][lvl]["checked"] += 1
                elems = _elements_from_mpc(primary_clean)
                pct_map = _element_pct_map_from_comp(comp_dict)
                missing_or_low = []
                for e in elems:
                    pct = pct_map.get(e)
                    if pct is None or pct <= 1.0:
                        missing_or_low.append(e)

                if not missing_or_low and elems:
                    checks[3][lvl]["passed"][repr(primary_clean)] += 1
                else:
                    fail_key = f"{repr(primary_clean)} -> <=1%/missing: {', '.join(missing_or_low) if missing_or_low else '(none)'}"
                    checks[3][lvl]["failed"][fail_key] += 1
                    check3_fail_files[lvl].setdefault(fail_key, set()).add(src_file)
                continue

            max_pair = _extract_max_element_from_comp(comp_dict)
            if max_pair is None:
                checks[3][lvl]["skipped"] += 1
                continue

            max_el, max_pct = max_pair

            checks[3][lvl]["checked"] += 1

            if primary_clean == "Other":
                checks[3][lvl]["passed"]["'Other'"] += 1
            elif primary_clean == max_el:
                checks[3][lvl]["passed"][repr(primary_clean)] += 1
            else:
                left = repr(primary_clean)
                pad = " " * max(1, align_width - len(left))
                fail_key = f"{left}{pad} -> max={max_el} ({max_pct:.3f}%)"
                checks[3][lvl]["failed"][fail_key] += 1
                check3_fail_files[lvl].setdefault(fail_key, set()).add(src_file)

    print("=== CHECK 1: If Multiple_Powders_Binary == False, Secondary/Tertiary primary elements must be blank ===\n")
    for lvl, _, _ in levels:
        _print_pass_fail(
            f"Check 1 — {lvl}_Powder_Primary_Element",
            checks[1][lvl]["passed"],
            checks[1][lvl]["failed"],
            checks[1][lvl]["checked"],
            checks[1][lvl]["skipped"],
        )

    print("=== CHECK 2: If non-blank, primary element must be in allowed element list ===\n")
    for lvl, _, _ in levels:
        _print_pass_fail(
            f"Check 2 — {lvl}_Powder_Primary_Element",
            checks[2][lvl]["passed"],
            checks[2][lvl]["failed"],
            checks[2][lvl]["checked"],
            checks[2][lvl]["skipped"],
        )

    print("=== CHECK 3: If non-blank primary AND composition is mapped JSON, max element must match primary ===\n")
    for lvl, _, _ in levels:
        _print_pass_fail(
            f"Check 3 — {lvl}_Powder_Primary_Element vs {lvl}_Powder_Chemical_Composition",
            checks[3][lvl]["passed"],
            checks[3][lvl]["failed"],
            checks[3][lvl]["checked"],
            checks[3][lvl]["skipped"],
        )
        if checks[3][lvl]["failed"]:
            print(f"Check 3 — {lvl} filenames for FAIL (element/composition mismatch):")
            for fail_key, _cnt in checks[3][lvl]["failed"].most_common():
                files = sorted(check3_fail_files.get(lvl, {}).get(fail_key, set()))
                for fn in files:
                    print(f"  {fail_key}  ::  {fn}")
            print()

    return {
        "exp_json_path": exp_json_path,
        "section": section,
        "allowed_elements": sorted(list(allowed)),
        "all_experiments": all_experiments,
        "checks": checks,
    }


result_primary_checks = evaluate_powder_primary_element_checks_jupyter(
    exp_json_path="../HUGO-CS/Dataset/AlternateVersions/Extracted-6-10-25_gt_remapped_swapped_deleted_filled_sorted_powderClass_grouped_imputed_blended_treatment.json",
    section="preSprayedProperties",
)


=== CHECK 1: If Multiple_Powders_Binary == False, Secondary/Tertiary primary elements must be blank ===

Check 1 — Majority_Powder_Primary_Element
  Checked: 3006 | Skipped: 1377
  PASS (Mapped):
      3006  N/A
  FAIL (Unmapped):
    (none)

Check 1 — Secondary_Powder_Primary_Element
  Checked: 3006 | Skipped: 1377
  PASS (Mapped):
      3006  ""
  FAIL (Unmapped):
    (none)

Check 1 — Tertiary_Powder_Primary_Element
  Checked: 3006 | Skipped: 1377
  PASS (Mapped):
      3006  ""
  FAIL (Unmapped):
    (none)

=== CHECK 2: If non-blank, primary element must be in allowed element list ===

Check 2 — Majority_Powder_Primary_Element
  Checked: 4291 | Skipped: 92
  PASS (Mapped):
      1126  'Al'
       965  'Cu'
       535  'Ti'
       517  'Ni'
       412  'Fe'
       161  'W'
        95  'Sn'
        91  'Cr'
        66  'Co'
        57  'Zn'
        57  'Ta'
        36  'MPC: Co, Cr, Fe, Mn, Ni'
        26  'B'
        21  'MPC: Co, Cr, Fe, Ni'
        21  'C'
        17  'Nb'
      

## Partitioning Mappings to indivudal Files

This code consumes a single string-mapping file with all string mappings, and saves three distinct files for weight, atomic and imputaiton.

In [ ]:
import json
import re
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, List, Tuple, Set


REPLACE_JSON = "../HUGO-CS/prompts/regExReplace2.json"
WEIGHT_KEY = "chemicalCompositionMapping"
ATOMIC_KEY = "chemicalCompositionMappingAtomic"
IMPUTE_KEY = "chemicalCompositionImputation"
OUT_DIR = Path("finalMappings")

OUT_WEIGHT = OUT_DIR / "weightComposition.json"
OUT_ATOMIC = OUT_DIR / "atomicComposition.json"
OUT_IMPUTE = OUT_DIR / "compositionImputation.json"
PB_KEY = "Majority_Powder_Pb_Percentage"
PD_KEY = "Majority_Powder_Pd_Percentage"
S_KEY = "Majority_Powder_S_Percentage"
TOP_KEY = ""
TOP_VALUE = "[V] Not Reported"
_NUM_RE = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")
_NUM_CHUNK_RE = re.compile(r"(\d+)")
_PCT_FIELD_RE = re.compile(r"^Majority_Powder_([A-Za-z]{1,2})_Percentage$")


def natural_key(s: str) -> Tuple[Any, ...]:
    parts = _NUM_CHUNK_RE.split(s)
    out: List[Any] = []
    for p in parts:
        if p.isdigit():
            out.append(int(p))
        else:
            out.append(p.casefold())
    return tuple(out)


def parse_pct_strict(x: Any, *, ctx: str) -> float:
    if x is None:
        return 0.0
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return 0.0
        m = _NUM_RE.search(s)
        if not m:
            raise ValueError(f"Unparsable percentage value at {ctx}: {x!r}")
        return float(m.group(0))
    raise ValueError(f"Unexpected percentage cell type at {ctx}: {type(x).__name__}")


def add_pd_between_pb_and_s(d: Dict[str, Any]) -> "OrderedDict[str, Any]":
    if PD_KEY in d:
        return OrderedDict(d.items())

    items = list(d.items())
    out: "OrderedDict[str, Any]" = OrderedDict()
    inserted = False
    has_pb = PB_KEY in d

    for k, v in items:
        if (not inserted) and (not has_pb) and (k == S_KEY):
            out[PD_KEY] = ""
            inserted = True

        out[k] = v

        if (not inserted) and (k == PB_KEY):
            out[PD_KEY] = ""
            inserted = True

    if not inserted:
        out[PD_KEY] = ""

    return out


def resolve_record_dict_strict(
    mapping: Dict[str, Any],
    record_key: str,
    *,
    seen: Set[str] | None = None
) -> Dict[str, Any]:
    if seen is None:
        seen = set()

    if record_key in seen:
        cycle = " -> ".join(list(seen) + [record_key])
        raise ValueError(f"Redirect cycle detected: {cycle}")
    seen.add(record_key)

    if record_key not in mapping:
        raise KeyError(f"Missing record key during redirect resolution: {record_key!r}")

    v = mapping[record_key]

    if isinstance(v, dict):
        return v

    if isinstance(v, str):
        target = v.strip()
        if target == "":
            raise ValueError(f"Record {record_key!r} redirects to empty string.")
        return resolve_record_dict_strict(mapping, target, seen=seen)

    raise TypeError(f"Record {record_key!r} has unsupported value type {type(v).__name__}.")


def extract_pairs_strict(v: Dict[str, Any], *, ctx_key: str) -> List[Tuple[str, float]]:
    pairs: List[Tuple[str, float]] = []
    for fname, fval in v.items():
        m = _PCT_FIELD_RE.match(fname)
        if not m:
            continue
        elem = m.group(1)
        if elem.casefold() == "other":
            continue
        pct = parse_pct_strict(fval, ctx=f"{ctx_key}.{fname}")
        pairs.append((elem, pct))

    if not pairs:
        raise ValueError(f"No percentage fields found for resolved record {ctx_key!r}")
    return pairs


def primary_element_strict(v: Dict[str, Any], *, ctx_key: str) -> Tuple[str, float]:
    pairs = extract_pairs_strict(v, ctx_key=ctx_key)
    max_pct = max(p for _, p in pairs)
    tied = sorted([e for e, p in pairs if p == max_pct], key=lambda s: s.casefold())
    return tied[0], max_pct


def composition_vector_strict(v: Dict[str, Any], *, ctx_key: str) -> Tuple[Tuple[str, float], ...]:
    pairs = extract_pairs_strict(v, ctx_key=ctx_key)
    return tuple(sorted(pairs, key=lambda ep: (ep[0].casefold(), ep[0], ep[1])))


def record_sort_key(mapping: Dict[str, Any], k: str) -> Tuple[Any, ...]:
    resolved = resolve_record_dict_strict(mapping, k)
    pe, pp = primary_element_strict(resolved, ctx_key=k)
    vec = composition_vector_strict(resolved, ctx_key=k)

    return (pe.casefold(), -pp, vec, natural_key(k))


def normalize_mapping_value_sorted(mapping: Dict[str, Any]) -> "OrderedDict[str, Any]":
    mapping[TOP_KEY] = TOP_VALUE
    keys_to_sort = [k for k in mapping.keys() if k != TOP_KEY]
    sorted_keys = sorted(keys_to_sort, key=lambda k: record_sort_key(mapping, k))

    out: "OrderedDict[str, Any]" = OrderedDict()
    out[TOP_KEY] = TOP_VALUE
    for k in sorted_keys:
        v = mapping[k]
        if isinstance(v, dict):
            out[k] = add_pd_between_pb_and_s(v)
        elif isinstance(v, str):
            out[k] = v 
        else:
            raise TypeError(f"Unsupported value type for record {k!r}: {type(v).__name__}")

    return out


def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
        f.write("\n")


def main() -> None:
    in_path = Path(REPLACE_JSON)
    with in_path.open("r", encoding="utf-8") as f:
        replace_dict = json.load(f)

    groups = [
        (WEIGHT_KEY, Path(OUT_WEIGHT)),
        (ATOMIC_KEY, Path(OUT_ATOMIC)),
        (IMPUTE_KEY, Path(OUT_IMPUTE)),
    ]

    for group_key, out_path in groups:
        mapping = replace_dict.get(group_key)
        if mapping is None:
            raise KeyError(f"Missing required group {group_key!r} in {in_path}")
        if not isinstance(mapping, dict):
            raise TypeError(f"Expected {group_key!r} to be a dict, got {type(mapping).__name__}")

        normalized = normalize_mapping_value_sorted(mapping)
        save_json(normalized, out_path)
        print(f"[WROTE] {group_key} -> {out_path} (items={len(normalized)})")


if __name__ == "__main__":
    main()



[WROTE] chemicalCompositionMapping -> finalMappings/weightComposition.json (items=867)
[WROTE] chemicalCompositionMappingAtomic -> finalMappings/atomicComposition.json (items=81)
[WROTE] chemicalCompositionImputation -> finalMappings/compositionImputation.json (items=96)


## Algorithmic Evaluation of Blend-Ratios for Multi-Powder Feedstocks

This code evaluates all the existing blend ratio mappings ensuring that they are 1) the right number of elements, 2) all usable numbers, and 3) sum to 100%

In [2]:
import json
import math
import re
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


EXPERIMENT_JSON = "../HUGO-CS/Dataset/AlternateVersions/Extracted-6-10-25_gt_remapped_swapped_deleted_filled_sorted.json"
REPLACE_JSON = "../HUGO-CS/prompts/regExReplace2.json"


SECTION = "preSprayedProperties"
RATIO_FIELD = "Powder_Blend_Ratio"


BLEND_MAP_KEY = "BlendRatioStandardization"

POWDER_NAME_FIELDS = [
    "Majority_Powder_Material_Name",
    "Secondary_Powder_Material_Name",
    "Tertiary_Powder_Material_Name",
]

NOT_REPORTED = "[V] Not Reported"
SUM_TOL = 1e-6
MIN_PART = 0.0
MAX_PART = 100.0
ZERO_EPS = 1e-12
OUT_JSON = "reviews/blend_ratio_algorithmic_report.json"
PCT_NUM_RE = re.compile(r"(\d+(?:\.\d+)?)\s*(?:wt%|vol%|at%|%)", re.IGNORECASE)

ALLOWED_IN_STD_RE = re.compile(r"^[\s0-9\.\:\,\/\;\-\+\(\)]+$")


def load_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Any, path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
        f.write("\n")
    Path(tmp).replace(path)


def safe_strip_str(x: Any) -> str:
    return x.strip() if isinstance(x, str) else ""


def is_reported_value(s: str) -> bool:
    ss = s.strip()
    return bool(ss) and ss != NOT_REPORTED


def extract_experiments(articles: Any) -> List[Tuple[str, int, Dict[str, Any]]]:
    out: List[Tuple[str, int, Dict[str, Any]]] = []

    if not isinstance(articles, list):
        return out

    for art in articles:
        if not isinstance(art, dict):
            continue

        filename = art.get("filename", "")
        if not isinstance(filename, str):
            filename = ""

        extracted = art.get("extractedText", {})
        if isinstance(extracted, str):
            try:
                extracted = json.loads(extracted)
            except json.JSONDecodeError:
                extracted = {}

        exps: List[Any] = []
        if isinstance(extracted, dict):
            exps = extracted.get("Experiments", []) or []
        elif isinstance(extracted, list):
            for item in extracted:
                if isinstance(item, dict):
                    exps.extend(item.get("Experiments", []) or [])

        if not isinstance(exps, list):
            continue

        for ei, exp in enumerate(exps):
            if isinstance(exp, dict):
                out.append((filename, ei, exp))

    return out


def powders_included(props: Dict[str, Any]) -> List[str]:
    powders: List[str] = []
    for f in POWDER_NAME_FIELDS:
        val = safe_strip_str(props.get(f, ""))
        if is_reported_value(val):
            powders.append(val)
    return powders


def sample_props_subset(props: Dict[str, Any]) -> Dict[str, Any]:
    keep = {
        RATIO_FIELD: props.get(RATIO_FIELD),
        "Majority_Powder_Material_Name": props.get("Majority_Powder_Material_Name"),
        "Secondary_Powder_Material_Name": props.get("Secondary_Powder_Material_Name"),
        "Tertiary_Powder_Material_Name": props.get("Tertiary_Powder_Material_Name"),
        "Majority_Powder_Chemical_Composition": props.get("Majority_Powder_Chemical_Composition"),
        "Secondary_Powder_Chemical_Composition": props.get("Secondary_Powder_Chemical_Composition"),
        "Tertiary_Powder_Chemical_Composition": props.get("Tertiary_Powder_Chemical_Composition"),
    }
    return keep


def parse_ratio_parts_from_standardized(s: str) -> Tuple[List[float], Optional[str]]:
    ss = safe_strip_str(s)
    if not ss:
        return [], "empty"

    if not ALLOWED_IN_STD_RE.match(ss):
        return [], "contains_disallowed_characters_or_letters"

    cleaned = ss.replace("(", " ").replace(")", " ")
    cleaned = re.sub(r"\s-\s", " ", cleaned)

    cleaned = cleaned.replace(":", " ").replace("/", " ").replace(",", " ").replace(";", " ")
    tokens = [t for t in cleaned.split() if t]

    parts: List[float] = []
    for t in tokens:
        try:
            parts.append(float(t))
        except ValueError:
            return [], f"unparsable_token:{t!r}"

    return parts, None


def parse_ratio_parts_from_raw_percent(s: str) -> Tuple[List[float], Optional[str]]:
    ss = safe_strip_str(s)
    if not ss:
        return [], "empty"
    matches = PCT_NUM_RE.findall(ss)
    if not matches:
        return [], "no_percent_adjacent_numbers"
    parts: List[float] = []
    for m in matches:
        try:
            parts.append(float(m))
        except ValueError:
            return [], f"unparsable_percent_number:{m!r}"
    return parts, None


def count_nonzero_parts(parts: List[float], eps: float = ZERO_EPS) -> int:
    return sum(1 for p in parts if abs(p) > eps)


@dataclass
class ExperimentRatioCheck:
    filename: str
    experiment_index: int

    ratio_raw: str
    ratio_mapped: Optional[str]
    ratio_used_for_check: str

    powders_included: List[str]
    ratio_parts: List[float]
    sum_parts: Optional[float]

    failures: List[str]
    notes: List[str]

    sample_props: Dict[str, Any]


def main() -> None:
    articles = load_json(EXPERIMENT_JSON)
    replace_dict = load_json(REPLACE_JSON)

    blend_map = replace_dict.get(BLEND_MAP_KEY, {})
    if not isinstance(blend_map, dict):
        raise TypeError(f"{BLEND_MAP_KEY!r} in {REPLACE_JSON} must be a dict.")

    blend_lower = {safe_strip_str(k).lower(): v for k, v in blend_map.items() if isinstance(k, str)}

    exp_rows = extract_experiments(articles)

    counts = Counter()
    results: List[ExperimentRatioCheck] = []

    for filename, ei, exp in exp_rows:
        props = exp.get(SECTION, {})
        if not isinstance(props, dict):
            continue

        raw = safe_strip_str(props.get(RATIO_FIELD, ""))
        if not raw:
            counts["missing_ratio"] += 1
            continue

        counts["nonempty_ratio"] += 1

        raw_key = raw.lower()
        mapped_val = blend_lower.get(raw_key)
        mapped = safe_strip_str(mapped_val) if isinstance(mapped_val, str) else None

        ratio_used = mapped or raw

        powders = powders_included(props)
        n_powders = len(powders)

        failures: List[str] = []
        notes: List[str] = []

        if mapped is not None:
            counts["mapped_by_blend_map"] += 1
        else:
            counts["unmapped_by_blend_map"] += 1
            failures.append("UNMAPPED_RAW_RATIO")
            notes.append(f"Raw ratio not found in regExReplace2[{BLEND_MAP_KEY!r}] (case-insensitive exact match).")

        parts: List[float] = []
        parse_notes: List[str] = []

        if mapped is not None:
            parts, err = parse_ratio_parts_from_standardized(ratio_used)
            if err:
                failures.append("BAD_MAPPED_RATIO_FORMAT")
                parse_notes.append(f"Mapped ratio {ratio_used!r} failed strict parse: {err}")
        else:
            parts, err = parse_ratio_parts_from_raw_percent(raw)
            if err:
                failures.append("UNPARSABLE_RAW_RATIO")
                parse_notes.append(f"Raw ratio {raw!r} could not be parsed via percent-adjacent rule: {err}")

        if parse_notes:
            notes.extend(parse_notes)

        s = sum(parts) if parts else None

        if not parts:
            counts["no_numeric_parts"] += 1
        else:
            nz_parts = count_nonzero_parts(parts, ZERO_EPS)
            if n_powders > 0 and nz_parts != n_powders:
                failures.append("POWDER_COUNT_MISMATCH")
                counts["powder_count_mismatch"] += 1
                notes.append(
                    f"Included powders={n_powders} but non-zero ratio parts={nz_parts} "
                    f"(raw parts={len(parts)}; zero_eps={ZERO_EPS})."
                )

            out_of_range = [p for p in parts if (p < MIN_PART - 1e-12) or (p > MAX_PART + 1e-12)]
            if out_of_range:
                failures.append("PARTS_OUT_OF_RANGE")
                counts["parts_out_of_range"] += 1
                notes.append(f"Ratio part(s) outside [{MIN_PART}, {MAX_PART}]: {out_of_range}")

            if s is not None and not math.isclose(s, 100.0, abs_tol=SUM_TOL):
                failures.append("SUM_NOT_100")
                counts["sum_not_100"] += 1
                notes.append(f"Sum(parts)={s} does not equal 100 within abs_tol={SUM_TOL}.")

        if failures:
            counts["experiments_with_failures"] += 1
            results.append(
                ExperimentRatioCheck(
                    filename=filename,
                    experiment_index=ei,
                    ratio_raw=raw,
                    ratio_mapped=mapped,
                    ratio_used_for_check=ratio_used,
                    powders_included=powders,
                    ratio_parts=parts,
                    sum_parts=s,
                    failures=failures,
                    notes=notes,
                    sample_props=sample_props_subset(props),
                )
            )
        else:
            counts["experiments_passing_checks"] += 1

    print("=== Blend Ratio Algorithmic Check Summary (regExReplace2-aware) ===")
    print(f"Total experiments: {len(exp_rows)}")
    print(f"Experiments with non-empty ratio: {counts['nonempty_ratio']}")
    print(f"Missing ratio (empty): {counts['missing_ratio']}")
    print("--- Mapping coverage ---")
    print(f"Mapped by {BLEND_MAP_KEY}: {counts['mapped_by_blend_map']}")
    print(f"Unmapped by {BLEND_MAP_KEY}: {counts['unmapped_by_blend_map']}")
    print("--- Failure counts (by type; may overlap per experiment) ---")
    for k, v in counts.most_common():
        if k in (
            "nonempty_ratio",
            "missing_ratio",
            "mapped_by_blend_map",
            "unmapped_by_blend_map",
            "experiments_with_failures",
            "experiments_passing_checks",
        ):
            continue
        print(f"{k}: {v}")
    print("--- Experiment-level totals ---")
    print(f"Experiments with ≥1 failure: {counts['experiments_with_failures']}")
    print(f"Experiments passing all checks: {counts['experiments_passing_checks']}")

    out = {
        "experiment_json": EXPERIMENT_JSON,
        "replace_json": REPLACE_JSON,
        "section": SECTION,
        "ratio_field": RATIO_FIELD,
        "blend_map_key": BLEND_MAP_KEY,
        "constraints": {
            "sum_tol": SUM_TOL,
            "min_part": MIN_PART,
            "max_part": MAX_PART,
            "zero_eps": ZERO_EPS,
        },
        "counts": dict(counts),
        "results": [asdict(r) for r in results],
    }

    save_json(out, OUT_JSON)
    print(f"\nSaved report to: {OUT_JSON}")
    print(f"Total experiments with failures recorded: {len(results)}")


if __name__ == "__main__":
    main()


=== Blend Ratio Algorithmic Check Summary (regExReplace2-aware) ===
Total experiments: 4383
Experiments with non-empty ratio: 1346
Missing ratio (empty): 3037
--- Mapping coverage ---
Mapped by BlendRatioStandardization: 1341
Unmapped by BlendRatioStandardization: 5
--- Failure counts (by type; may overlap per experiment) ---
no_numeric_parts: 4
sum_not_100: 1
powder_count_mismatch: 1
--- Experiment-level totals ---
Experiments with ≥1 failure: 6
Experiments passing all checks: 1340

Saved report to: reviews/blend_ratio_algorithmic_report.json
Total experiments with failures recorded: 6


## LLM Review of mappings:

In [3]:
import json
import os
import time
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from openai import OpenAI



EXPERIMENT_JSON = "../HUGO-CS/Dataset/AlternateVersions/Extracted-6-10-25_gt_remapped_swapped_deleted_filled_sorted.json"
REPLACE_JSON = "../HUGO-CS/prompts/regExReplace2.json"

SECTION = "preSprayedProperties"
RATIO_FIELD = "Powder_Blend_Ratio"
BLEND_MAP_KEY = "BlendRatioStandardization"

POWDER_NAME_FIELDS = [
    "Majority_Powder_Material_Name",
    "Secondary_Powder_Material_Name",
    "Tertiary_Powder_Material_Name",
]

OPENAI_TOKEN_PATH = (
    "../openaiToken.txt"
)

MODEL = "o4-mini"
REASONING_EFFORT = "high"

MAX_WORKERS = 8
MAX_RETRIES = 5
BACKOFF_SECONDS = 2.0

OUT_JSON = "reviews/blend_ratio_llm_review.json"

def load_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Any, path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
        f.write("\n")
    os.replace(tmp, path)


def clean_json_fence(text: str) -> str:
    t = (text or "").strip()
    if t.startswith("```"):
        t = t.replace("```json", "").replace("```", "").strip()
    return t


def try_parse_json(text: str) -> Optional[Dict[str, Any]]:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


def safe_strip_str(x: Any) -> str:
    return x.strip() if isinstance(x, str) else ""


def extract_experiments(articles: Any) -> List[Tuple[str, int, Dict[str, Any]]]:
    out: List[Tuple[str, int, Dict[str, Any]]] = []

    if not isinstance(articles, list):
        return out

    for art in articles:
        if not isinstance(art, dict):
            continue

        filename = art.get("filename", "")
        if not isinstance(filename, str):
            filename = ""

        extracted = art.get("extractedText", {})
        if isinstance(extracted, str):
            try:
                extracted = json.loads(extracted)
            except json.JSONDecodeError:
                extracted = {}

        exps: List[Any] = []
        if isinstance(extracted, dict):
            exps = extracted.get("Experiments", []) or []
        elif isinstance(extracted, list):
            for item in extracted:
                if isinstance(item, dict):
                    exps.extend(item.get("Experiments", []) or [])

        if not isinstance(exps, list):
            continue

        for ei, exp in enumerate(exps):
            if isinstance(exp, dict):
                out.append((filename, ei, exp))

    return out


def props_subset(props: Dict[str, Any]) -> Dict[str, Any]:
    keep: Dict[str, Any] = {
        RATIO_FIELD: props.get(RATIO_FIELD),
    }
    for f in POWDER_NAME_FIELDS:
        keep[f] = props.get(f)
    keep["Majority_Powder_Chemical_Composition"] = props.get("Majority_Powder_Chemical_Composition")
    keep["Secondary_Powder_Chemical_Composition"] = props.get("Secondary_Powder_Chemical_Composition")
    keep["Tertiary_Powder_Chemical_Composition"] = props.get("Tertiary_Powder_Chemical_Composition")
    return keep


def build_unique_combos(
    exp_rows: List[Tuple[str, int, Dict[str, Any]]],
    blend_lower: Dict[str, Any],
) -> List[Dict[str, Any]]:
    counter = Counter()
    samples: Dict[Tuple[str, str, str, str, str], Dict[str, Any]] = {}

    for filename, ei, exp in exp_rows:
        props = exp.get(SECTION, {})
        if not isinstance(props, dict):
            continue

        ratio_raw = safe_strip_str(props.get(RATIO_FIELD, ""))
        if not ratio_raw:
            continue

        mapped_val = blend_lower.get(ratio_raw.lower())
        ratio_mapped = safe_strip_str(mapped_val) if isinstance(mapped_val, str) else ""

        names = [safe_strip_str(props.get(f, "")) for f in POWDER_NAME_FIELDS]
        majority, secondary, tertiary = names[0], names[1], names[2]

        key = (ratio_raw, ratio_mapped, majority, secondary, tertiary)

        counter[key] += 1
        if key not in samples:
            samples[key] = {
                "filename": filename,
                "experiment_index": ei,
                "props": props_subset(props),
            }

    combos: List[Dict[str, Any]] = []
    for (ratio_raw, ratio_mapped, majority, secondary, tertiary), freq in counter.most_common():
        combos.append(
            {
                "frequency": freq,
                "ratio_raw": ratio_raw,
                "ratio_mapped": ratio_mapped if ratio_mapped != "" else None,
                "majority_powder": majority,
                "secondary_powder": secondary,
                "tertiary_powder": tertiary,
                "sample": samples[(ratio_raw, ratio_mapped, majority, secondary, tertiary)],
            }
        )
    return combos


def run_llm_assessment(client: OpenAI, combo: Dict[str, Any]) -> Dict[str, Any]:
    """
    Returns:
      {
        "rating": 1|2|3 (or None if parse failed),
        "verdict": str,
        "explanation": str,
        "recommended_correction": str,
        "raw": str,
        "tokens": int
      }
    """
    system = (
        "You are a meticulous QA reviewer for a materials/cold-spray dataset. "
        "You will be given one UNIQUE combination of:\n"
        "- Powder blend ratio string (raw)\n"
        "- Its mapped/standardized string from regExReplace2['BlendRatioStandardization'] (may be missing)\n"
        "- Powder material name strings (majority/secondary/tertiary)\n"
        "- Optionally, powder composition strings\n\n"
        "Your job: judge whether the mapping is correct and internally consistent.\n\n"
        "Rules:\n"
        "- Treat the mapping as a standardized ratio format. Preferred output: colon-separated numeric parts "
        "(e.g., '95.4:4.6', '60:40', '100:0'). Decimals are allowed.\n"
        "- The mapping should represent the blend among reported powders. Default assumption: ordering matches "
        "Majority then Secondary then Tertiary when multiple powders are present.\n"
        "- Numeric parts should typically sum to 100 (small rounding differences acceptable).\n"
        "- Explicit zeros are allowed; a mapping like '100:0' is acceptable for a single reported powder.\n"
        "- If the raw ratio looks like an alloy designation rather than a blend ratio (e.g., 'WC-17Co' or 'CuSn10' "
        "embedded in the ratio string), do NOT treat those embedded numbers as ratio parts. Focus on the raw ratio "
        "percentages/phrasing.\n"
        "- If mapping is missing (unmapped), propose what it should be if the raw ratio clearly specifies the fractions.\n"
        "- Do not invent information not present. If ambiguous, choose rating 2 and propose the best reasonable correction.\n\n"
        "Return ONLY valid JSON with this schema:\n"
        "{\n"
        '  \"rating\": 1|2|3,\n'
        '  \"verdict\": \"Fully Correct\" | \"Potentially incorrect\" | \"Definitely wrong\",\n'
        '  \"explanation\": \"Concise, specific justification.\",\n'
        '  \"recommended_correction\": \"If rating!=1, provide a corrected mapped ratio string (e.g., 60:40). '
        'If rating==1, return an empty string.\"\n'
        "}\n\n"
        "Rating definitions:\n"
        "1 = Fully Correct: mapping is consistent.\n"
        "2 = Potentially incorrect: plausible but unclear/ambiguous; explain and propose correction.\n"
        "3 = Definitely wrong: clear contradiction; propose correction.\n"
    )

    user = (
        "Evaluate this unique blend-ratio mapping.\n\n"
        f"Frequency in dataset: {combo.get('frequency')}\n\n"
        f"Raw ratio string:\n{combo.get('ratio_raw')!r}\n\n"
        f"Mapped ratio string (from BlendRatioStandardization):\n{combo.get('ratio_mapped')!r}\n\n"
        "Powders:\n"
        f"  Majority:  {combo.get('majority_powder')!r}\n"
        f"  Secondary: {combo.get('secondary_powder')!r}\n"
        f"  Tertiary:  {combo.get('tertiary_powder')!r}\n\n"
        "Sample preSprayedProperties fields (subset):\n"
        f"{json.dumps((combo.get('sample') or {}).get('props', {}), indent=2, ensure_ascii=False)}\n"
    )

    last_err: Optional[Exception] = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                reasoning_effort=REASONING_EFFORT,
            )

            raw = resp.choices[0].message.content or ""
            tokens = int(getattr(resp.usage, "total_tokens", 0) or 0)

            cleaned = clean_json_fence(raw)
            parsed = try_parse_json(cleaned)

            if parsed is None:
                return {
                    "rating": None,
                    "verdict": None,
                    "explanation": None,
                    "recommended_correction": None,
                    "raw": cleaned,
                    "tokens": tokens,
                    "error": "invalid_json_from_model",
                }

            rating = parsed.get("rating")
            verdict = parsed.get("verdict")
            explanation = parsed.get("explanation")
            rec = parsed.get("recommended_correction")

            if rating not in (1, 2, 3):
                rating = None

            if rec is None:
                rec = ""
            elif not isinstance(rec, str):
                rec = str(rec)

            return {
                "rating": rating,
                "verdict": verdict,
                "explanation": explanation,
                "recommended_correction": rec,
                "raw": cleaned,
                "tokens": tokens,
            }

        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(BACKOFF_SECONDS * attempt)
                continue
            raise RuntimeError(f"LLM call failed after {MAX_RETRIES} attempts: {e}") from e

    raise RuntimeError(f"Unexpected failure: {last_err}")


def main() -> None:
    api_key = Path(OPENAI_TOKEN_PATH).read_text(encoding="utf-8").strip()
    os.environ["OPENAI_API_KEY"] = api_key
    client = OpenAI(api_key=api_key)

    articles = load_json(EXPERIMENT_JSON)
    replace_dict = load_json(REPLACE_JSON)

    blend_map = replace_dict.get(BLEND_MAP_KEY, {})
    if not isinstance(blend_map, dict):
        raise TypeError(f"{BLEND_MAP_KEY!r} in {REPLACE_JSON} must be a dict.")

    blend_lower = {safe_strip_str(k).lower(): v for k, v in blend_map.items() if isinstance(k, str)}

    exp_rows = extract_experiments(articles)
    combos = build_unique_combos(exp_rows, blend_lower)

    print(f"Built {len(combos)} unique (powders + raw ratio + mapped ratio) combos to review.")

    results: List[Dict[str, Any]] = []
    token_total = 0

    def worker(idx: int, combo: Dict[str, Any]) -> Dict[str, Any]:
        assessment = run_llm_assessment(client, combo)
        row = {
            "idx": idx,
            "frequency": combo.get("frequency"),
            "ratio_raw": combo.get("ratio_raw"),
            "ratio_mapped": combo.get("ratio_mapped"),
            "majority_powder": combo.get("majority_powder"),
            "secondary_powder": combo.get("secondary_powder"),
            "tertiary_powder": combo.get("tertiary_powder"),
            "sample": combo.get("sample"),
            "rating": assessment.get("rating"),
            "verdict": assessment.get("verdict"),
            "explanation": assessment.get("explanation"),
            "recommended_correction": assessment.get("recommended_correction"),
            "tokens": assessment.get("tokens"),
            "raw": assessment.get("raw"),
        }
        if "error" in assessment:
            row["error"] = assessment["error"]
        return row

    jobs = list(enumerate(combos, start=1))
    job_iter = iter(jobs)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {}

        while len(futures) < MAX_WORKERS:
            try:
                idx, combo = next(job_iter)
            except StopIteration:
                break
            fut = pool.submit(worker, idx, combo)
            futures[fut] = idx

        while futures:
            done, _ = wait(futures, return_when=FIRST_COMPLETED)
            for fut in done:
                idx = futures.pop(fut)
                row = fut.result()

                token_total += int(row.get("tokens", 0) or 0)
                results.append(row)

                expl = row.get("explanation") or row.get("raw") or ""
                expl_preview = " ".join(str(expl).split())
                if len(expl_preview) > 220:
                    expl_preview = expl_preview[:220] + "..."

                rec = row.get("recommended_correction") or ""
                rec_preview = " ".join(str(rec).split())
                if len(rec_preview) > 120:
                    rec_preview = rec_preview[:120] + "..."

                print(f"[{idx}/{len(combos)}] rating={row.get('rating')} verdict={row.get('verdict')} tokens_total={token_total}")
                print(f"  Raw:    {row.get('ratio_raw')!r}")
                print(f"  Mapped: {row.get('ratio_mapped')!r}")
                print(f"  Powders: M={row.get('majority_powder')!r}, S={row.get('secondary_powder')!r}, T={row.get('tertiary_powder')!r}")
                if rec_preview:
                    print(f"  RECOMMENDED: {rec_preview}")
                print(f"  NOTE: {expl_preview}")
                print("-" * 80)

            while len(futures) < MAX_WORKERS:
                try:
                    idx, combo = next(job_iter)
                except StopIteration:
                    break
                fut = pool.submit(worker, idx, combo)
                futures[fut] = idx

    results_sorted = sorted(results, key=lambda r: r.get("idx", 10**18))

    out_obj = {
        "experiment_json": EXPERIMENT_JSON,
        "replace_json": REPLACE_JSON,
        "blend_map_key": BLEND_MAP_KEY,
        "section": SECTION,
        "ratio_field": RATIO_FIELD,
        "model": MODEL,
        "reasoning_effort": REASONING_EFFORT,
        "total_tokens": token_total,
        "results": results_sorted,
    }
    save_json(out_obj, OUT_JSON)
    print(f"Saved {len(results_sorted)} combo reviews to {OUT_JSON}. Total tokens: {token_total}")


if __name__ == "__main__":
    main()


Built 749 unique (powders + raw ratio + mapped ratio) combos to review.
[1/749] rating=1 verdict=Fully Correct tokens_total=782
  Raw:    '70 wt% Copper, 30 wt% Alumina'
  Mapped: '70:30'
  Powders: M='Copper', S='Alumina', T=''
  NOTE: The standardized ratio 70:30 accurately reflects 70 wt% Copper (majority) and 30 wt% Alumina (secondary), summing to 100%.
--------------------------------------------------------------------------------
[5/749] rating=1 verdict=Fully Correct tokens_total=1544
  Raw:    '95 wt% WC, 5 wt% Ni'
  Mapped: '95:5'
  Powders: M='WC', S='Ni', T=''
  NOTE: The mapping “95:5” accurately reflects the 95 wt% WC and 5 wt% Ni blend in majority:secondary order.
--------------------------------------------------------------------------------
[6/749] rating=1 verdict=Fully Correct tokens_total=2333
  Raw:    '60 wt% B4C, 40 wt% Al'
  Mapped: '60:40'
  Powders: M='B4C', S='Al', T=''
  NOTE: The raw ratio clearly specifies 60 wt% B4C and 40 wt% Al, and the mapping “60:40”

KeyboardInterrupt: 